# 第二階段：找出一批圓

**輸入**：圖片路徑 + 第一階段估出的 r
**輸出**：一批圓，每個圓有 x, y, r, score（**原圖尺度**）

## 怎麼用
1. 改「設定」那格的 `IMAGE_PATH`
2. `R` 可以留 `None`：會自動讀第一階段存的結果（`.tmp/<圖片檔名>/summary.json`）；
   也可以直接填一個數字
3. 從上到下依序執行

## 做了什麼
1. **縮放**：倍率 s = 40 ÷ r，讓所有圓都變成半徑 40（共同尺度）
2. 在縮放後的圖上，跑跟第一階段一樣的步驟 1～4
   （差別：Hough 只找半徑 40 附近的圓，內部小圓一開始就不會被提名）
3. 座標除以 s，換回原圖

這裡的圓心是「大概的位置」，因為是在縮小的圖上找的。第三階段會回到原圖修準。

結果存在 `.tmp/<圖片檔名>/detect/summary.json`，第三個 notebook 會自動讀取。

In [ ]:
import os, json, csv
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 如果上面出現 ModuleNotFoundError，先執行：
#   pip install opencv-python numpy matplotlib

## 設定（只要改這裡）

In [ ]:
IMAGE_PATH = "your_image.jpg"   # ← 改成你的圖片路徑
R = None                        # ← None = 自動讀第一階段的結果；也可以直接填數字，例如 130.3
TMP_DIR = ".tmp"

In [ ]:
name = os.path.splitext(os.path.basename(IMAGE_PATH))[0]
if R is None:
    stage1 = os.path.join(TMP_DIR, name, "summary.json")
    R = json.load(open(stage1))["r"]
    print(f"從 {stage1} 讀到 r = {R:.1f}")
else:
    print(f"使用手動填的 r = {R}")

## 參數
因為所有圖都先縮放成「圓半徑 = 40」，這些數字對每張圖都一樣，設計時定一次就好。

**給 IC 影像的建議**：圓大小一致，可以把 `RADIUS_RANGE` 改成 `(0.8, 1.2)`，結果會更乾淨。

In [ ]:
PARAMS = {
    # 共同尺度：所有圖都縮放到圓的半徑 = 這個數字（像素）
    "R_STD": 40,

    # Hough 只找半徑在 R_STD 的幾倍之間的圓
    # 團藻本身大小差很多，所以用 0.6～1.4；
    # IC 圓大小一致，建議改成 (0.8, 1.2)，結果會更乾淨
    "RADIUS_RANGE": (0.6, 1.4),

    # 模糊 = R_STD ÷ 這個數字（半徑的 1/20）
    "BLUR_DIVISOR": 20,

    # Hough：兩個圓心至少隔 R_STD 的幾倍
    "HOUGH_MIN_DIST": 0.25,
    # Hough：至少幾票才算候選（故意很低，寧可多提名，靠打分數把關）
    "HOUGH_VOTES": 12,
    # Hough：內部 Canny 邊緣門檻
    "HOUGH_CANNY": 80,

    # 打分數：mag 要比多少 % 的像素強，才算「強邊緣」
    "EDGE_PERCENTILE": 65,
    # 打分數：邊緣箭頭跟半徑方向夾角的 cos 要大於多少（0.8 ≈ 37 度以內）
    "DIRECTION_COS": 0.8,
    # 打分數：至少幾分才算真圓
    "MIN_SCORE": 0.6,

    # NMS：圓心距離 < OVERLAP ×（兩半徑相加）→ 重疊太多
    "OVERLAP": 0.6,
    # NMS：圓心距離 < INSIDE × 較大的半徑 → 包在裡面
    "INSIDE": 0.85,
}

## 共用工具
跟第一階段一模一樣的四個工具：邊緣、打分數、微調、NMS，再加上畫圖。

In [ ]:
def compute_edges(gray, blur):
    """
    算出每個像素的邊緣資訊（三張跟 gray 一樣大的表格）：
        gx   左右方向亮度變化（往右變亮 = 正，變暗 = 負）
        gy   上下方向亮度變化（往下變亮 = 正）
        mag  邊緣強度 √(gx² + gy²)
    (gx, gy) 合起來是一個箭頭，指向「變亮的方向」。
    """
    b = cv2.GaussianBlur(gray, (0, 0), blur)
    gx = cv2.Sobel(b, cv2.CV_32F, 1, 0)       # 一定要 CV_32F，才放得下負數
    gy = cv2.Sobel(b, cv2.CV_32F, 0, 1)
    return gx, gy, cv2.magnitude(gx, gy)

In [ ]:
def score_circle(edges, thr, x, y, r, p, n_points=72, band_px=2):
    """
    沿圓周取 72 個檢查點，每個點問兩個問題：
        一、這裡有強邊緣嗎？（mag > thr）
        二、邊緣箭頭有沿著半徑方向嗎？（指向或背向圓心，不能斜）
    兩個都是「是」才算一分。分數 = 得分點 ÷ 檢查點（0～1）。

    band_px：每個點往內、往外各多看幾格，容許半徑有一點點誤差。
    落在圖外的點不算（不加分也不扣分），被邊界切到的圓才不會吃虧。
    """
    gx, gy, mag = edges
    h, w = mag.shape
    ang = np.linspace(0, 2 * np.pi, n_points, endpoint=False)
    ca, sa = np.cos(ang)[:, None], np.sin(ang)[:, None]
    rs = r + np.arange(-band_px, band_px + 1)[None, :]
    X = np.round(x + ca * rs).astype(int)
    Y = np.round(y + sa * rs).astype(int)

    inside = ((X >= 0) & (X < w) & (Y >= 0) & (Y < h)).all(axis=1)
    if inside.sum() < n_points * 0.5:
        return 0.0

    Xc, Yc = np.clip(X, 0, w - 1), np.clip(Y, 0, h - 1)
    m = mag[Yc, Xc]
    radial = np.abs(gx[Yc, Xc] * ca + gy[Yc, Xc] * sa) / (m + 1e-6)
    hit = ((m > thr) & (radial > p["DIRECTION_COS"])).any(axis=1)
    return float((hit & inside).sum() / inside.sum())

In [ ]:
def refine_center(mag, x, y, r, n_rays=90):
    """
    Hough 給的圓心常常偏一點，打分數前先微調，免得真圓被冤枉扣分。
    從圓心射出 90 條線，每條只在 0.85r～1.15r 的環帶裡找最強的邊緣點，
    再用這些點擬合一個最貼合的圓（最小平方法）。修太多就不採用。
    """
    h, w = mag.shape
    rs = np.linspace(0.85 * r, 1.15 * r, 31)
    pts, wts = [], []
    for a in np.linspace(0, 2 * np.pi, n_rays, endpoint=False):
        X = np.round(x + rs * np.cos(a)).astype(int)
        Y = np.round(y + rs * np.sin(a)).astype(int)
        if not ((X >= 0) & (X < w) & (Y >= 0) & (Y < h)).all():
            continue
        v = mag[Y, X]
        j = int(np.argmax(v))
        pts.append((X[j], Y[j]))
        wts.append(v[j])
    if len(pts) < n_rays * 0.5:
        return x, y, r
    P = np.array(pts, float)[np.array(wts) > np.percentile(wts, 25)]
    A = np.c_[2 * P[:, 0], 2 * P[:, 1], np.ones(len(P))]
    cx, cy, c = np.linalg.lstsq(A, (P ** 2).sum(1), rcond=None)[0]
    cr = np.sqrt(c + cx ** 2 + cy ** 2)
    if np.hypot(cx - x, cy - y) > 0.2 * r or abs(cr - r) > 0.2 * r:
        return x, y, r
    return cx, cy, cr

In [ ]:
def nms(scored, p):
    """
    依「分數 × √半徑」由高到低排隊，一個一個選。
    跟已選的圓衝突就淘汰：
        - 重疊太多：距離 < OVERLAP × (r1 + r2)
        - 包在裡面：距離 < INSIDE × max(r1, r2)   ← 內部小圓就是這樣被淘汰的
    輸入輸出都是 list of dict（含 x, y, r, score）。
    """
    order = sorted(scored, key=lambda c: -(c["score"] * np.sqrt(c["r"])))
    kept = []
    for c in order:
        ok = True
        for k in kept:
            d = np.hypot(c["x"] - k["x"], c["y"] - k["y"])
            if d < p["OVERLAP"] * (c["r"] + k["r"]) or d < p["INSIDE"] * max(c["r"], k["r"]):
                ok = False
                break
        if ok:
            kept.append(c)
    return kept

In [ ]:
def _draw(base_bgr, circles, path, color_fn, thickness=1, note=None):
    img = base_bgr.copy()
    for c in circles:
        cv2.circle(img, (int(round(c["x"])), int(round(c["y"]))), max(1, int(round(c["r"]))),
                   color_fn(c), thickness)
    if note:
        cv2.rectangle(img, (0, 0), (img.shape[1], 26), (0, 0, 0), -1)
        cv2.putText(img, note, (6, 19), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)
    cv2.imwrite(path, img)


def show(path, title=None, width=10):
    """在 notebook 裡顯示一張存好的圖"""
    img = cv2.imread(path)
    if img is None:
        print("找不到圖：", path); return
    h, w = img.shape[:2]
    if title:
        print(title)   # 用 print 顯示標題（matplotlib 預設字型沒有中文）
    plt.figure(figsize=(width, width * h / w))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

## 主函式

In [ ]:
def detect_circles(image_path, r, tmp_dir=".tmp", params=None):
    """
    輸入：
        image_path  圖片路徑
        r           這張圖的圓大概多大（原圖像素），通常來自 estimate_radius()
        tmp_dir     中間資料存在哪裡
        params      可以傳 dict 覆蓋部分參數，例如 {"RADIUS_RANGE": (0.8, 1.2)}

    輸出：
        list of dict，每個圓一筆：{"x", "y", "r", "score"}，都是原圖像素。
        一個都找不到就回傳空 list。
    """
    p = dict(PARAMS, **(params or {}))
    R = p["R_STD"]

    name = os.path.splitext(os.path.basename(image_path))[0]
    out = os.path.join(tmp_dir, name, "detect")
    os.makedirs(out, exist_ok=True)

    gray_full = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if gray_full is None:
        raise FileNotFoundError(image_path)
    H, W = gray_full.shape

    # ---------- 1. 縮放：讓圓變成半徑 R ----------
    s = R / r
    # 縮小用 INTER_AREA，放大用 INTER_CUBIC，畫質比較好
    method = cv2.INTER_AREA if s < 1 else cv2.INTER_CUBIC
    gray = cv2.resize(gray_full, None, fx=s, fy=s, interpolation=method)
    base = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    cv2.imwrite(os.path.join(out, "01_scaled.png"), gray)

    # 模糊固定是半徑的 1/20。因為 R 固定，所以對每張圖都一樣
    blur = R / p["BLUR_DIVISOR"]

    # ---------- 2-1. 邊緣 ----------
    edges = compute_edges(gray, blur)
    mag = edges[2]
    cv2.imwrite(os.path.join(out, "02_edges.png"),
                np.clip(mag / np.percentile(mag, 99) * 255, 0, 255).astype(np.uint8))
    thr = float(np.percentile(mag, p["EDGE_PERCENTILE"]))

    # ---------- 2-2. Hough：只找 R 附近大小的圓 ----------
    lo, hi = p["RADIUS_RANGE"]
    found = cv2.HoughCircles(
        cv2.GaussianBlur(gray, (0, 0), blur), cv2.HOUGH_GRADIENT, dp=1,
        minDist=p["HOUGH_MIN_DIST"] * R, param1=p["HOUGH_CANNY"], param2=p["HOUGH_VOTES"],
        minRadius=int(lo * R), maxRadius=int(hi * R),
    )
    cands = [] if found is None else found[0].tolist()
    rows = [{"x": x, "y": y, "r": rr, "score": None, "stage": "hough"} for x, y, rr in cands]
    _draw(base, rows, os.path.join(out, "03_candidates.jpg"), lambda c: (0, 0, 255),
          note=f"Hough candidates (r {lo * R:.0f}-{hi * R:.0f}): {len(rows)}")

    # ---------- 2-3. 打分數 ----------
    # 先快速粗篩（省時間），再微調圓心，再正式打分數
    for c in rows:
        quick = score_circle(edges, thr, c["x"], c["y"], c["r"], p,
                             n_points=36, band_px=max(2, int(0.06 * c["r"])))
        if quick < p["MIN_SCORE"] - 0.15:
            c["stage"] = "dropped_quick_check"
            continue
        c["x"], c["y"], c["r"] = map(float, refine_center(mag, c["x"], c["y"], c["r"]))
        c["score"] = score_circle(edges, thr, c["x"], c["y"], c["r"], p)
        c["stage"] = "passed_score" if c["score"] >= p["MIN_SCORE"] else "dropped_low_score"
    passed = [c for c in rows if c["stage"] == "passed_score"]
    _draw(base, passed, os.path.join(out, "04_passed_score.jpg"),
          lambda c: (0, int(255 * c["score"]), int(255 * (1 - c["score"]))),
          note=f"passed score >= {p['MIN_SCORE']}: {len(passed)}")

    # ---------- 2-4. NMS ----------
    kept = nms(passed, p)
    for c in passed:
        c["stage"] = "kept_after_nms" if c in kept else "dropped_nms"
    _draw(base, kept, os.path.join(out, "05_after_nms.jpg"), lambda c: (0, 220, 0), 2,
          note=f"after NMS: {len(kept)}")

    # ---------- 3. 換回原圖 ----------
    # 縮放時乘了 s，現在除以 s 還原
    for c in rows:
        c["x"], c["y"], c["r"] = c["x"] / s, c["y"] / s, c["r"] / s

    circles = [{"x": round(c["x"], 2), "y": round(c["y"], 2), "r": round(c["r"], 2),
                "score": round(c["score"], 3)} for c in kept]

    # 存檔
    base_full = cv2.imread(image_path)
    _draw(base_full, circles, os.path.join(out, "06_result.jpg"), lambda c: (0, 0, 255),
          max(2, int(max(H, W) / 400)), note=f"detected circles: {len(circles)}")
    with open(os.path.join(out, "candidates.csv"), "w", newline="") as f:
        wr = csv.writer(f)
        wr.writerow(["x", "y", "r", "score", "stage"])
        for c in rows:
            wr.writerow([round(c["x"], 2), round(c["y"], 2), round(c["r"], 2),
                         "" if c["score"] is None else round(c["score"], 3), c["stage"]])
    with open(os.path.join(out, "summary.json"), "w") as f:
        json.dump({
            "image": image_path,
            "image_size": [W, H],
            "r_input": r,
            "scale": s,
            "scaled_size": [gray.shape[1], gray.shape[0]],
            "params": p,
            "counts": {
                "hough_candidates": len(rows),
                "passed_score": len(passed),
                "after_nms": len(kept),
            },
            "circles": circles,
        }, f, indent=1)

    return circles

## 執行

In [ ]:
circles = detect_circles(IMAGE_PATH, R, tmp_dir=TMP_DIR)

out = os.path.join(TMP_DIR, name, "detect")
summary = json.load(open(os.path.join(out, "summary.json")))
print(f"縮放倍率 s = {summary['scale']:.3f}，縮放後大小 {summary['scaled_size']}")
print("每一步剩幾個圓：", summary["counts"])
print(f"找到 {len(circles)} 個圓，前 5 個：")
for c_ in circles[:5]:
    print("  ", c_)

## 看中間結果

In [ ]:
show(os.path.join(out, "03_candidates.jpg"), "Hough 提名的候選圓（縮放後的圖）", width=6)
show(os.path.join(out, "05_after_nms.jpg"), "NMS 之後（縮放後的圖）", width=6)
show(os.path.join(out, "06_result.jpg"), "最後結果（畫回原圖）")